<a href="https://colab.research.google.com/github/neel0086/MachineLearning/blob/main/LogisticRegression/Heartdiseaseprediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import kagglehub
import numpy as np
import pandas as pd

In [2]:
path = kagglehub.dataset_download(
    "shyamnadhs/heart-disease-prediction-dataset"
)

print("Dataset Path:", path)

100%|██████████| 16.1k/16.1k [00:00<00:00, 22.1MB/s]

Extracting files...
Dataset Path: /root/.cache/kagglehub/datasets/shyamnadhs/heart-disease-prediction-dataset/versions/3


In [3]:
df = pd.read_csv(os.path.join(path, "disease_prediction.csv"))

print(df.head())
print(df.columns.tolist())

   patient_id  age  gender  glucose_mg_dl  cholesterol_mg_dl  systolic_bp  \
0           1   32    Male            101                235          152   
1           2   31    Male            124                191          134   
2           3   45    Male             57                141          114   
3           4   75  Female             69                268          120   
4           5   53    Male            107                163          131   

   diastolic_bp   bmi  heart_rate smoking alcohol_consumption  \
0            79  28.5          73      No                 Yes   
1            77  33.9          71      No                 Yes   
2            71  27.2          79     Yes                 Yes   
3            82  21.5          61     Yes                 Yes   
4            75  23.3          73     Yes                  No   

  physical_activity family_history disease  
0               Low            Yes     Yes  
1               Low            Yes     Yes  
2          

In [4]:
df.drop(columns=["patient_id"], inplace=True, errors="ignore")

TARGET_COLUMN = "disease"

X = df.drop(columns=[TARGET_COLUMN])

Y = df[TARGET_COLUMN].map({
    "No":0,
    "Yes":1
})

In [5]:
indices = np.random.permutation(len(X))

#Gives shuffle dataset as per above random permutation
X = X.iloc[indices].reset_index(drop=True)
Y = Y.iloc[indices].reset_index(drop=True)

In [6]:
#TEST TRAIN SPLIT
split = int(len(X)*0.8)

X_train = X.iloc[:split].copy()
X_test = X.iloc[split:].copy()
Y_train = Y.iloc[:split].copy()
Y_test = Y.iloc[split:].copy()


In [7]:
categorical_columns = X_train.select_dtypes(include="object").columns

X_train = pd.get_dummies(X_train, columns=categorical_columns)

X_test = pd.get_dummies(X_test, columns=categorical_columns)

X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

In [8]:
#Feature scaling standarization
mean = X_train.mean()
std = X_train.std()

X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

X_train = X_train.to_numpy(dtype=float)
X_test = X_test.to_numpy(dtype=float)
Y_train = Y_train.to_numpy(dtype=float)
Y_test = Y_test.to_numpy(dtype=float)

In [9]:
#Model parameters
weights = np.zeros(X_train.shape[1])

bias = 0

learning_rate = 0.01
epochs = 5000

In [10]:
def sigmoid(z):
  return 1/(1+np.exp(-z))

In [11]:
def binary_cross_entropy(y_true, y_pred):
  epsilon = 1e-15 # Typo corrected from 'eplison'

  y_pred = np.clip(y_pred, epsilon, 1 - epsilon) # Using 'epsilon'
  return np.mean(-(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))) # Corrected parenthesis

In [12]:
for epoch in range(epochs):
  z = X_train@weights + bias

  prediction = sigmoid(z)

  current_loss = binary_cross_entropy(Y_train, prediction)
  error = prediction - Y_train

  dw = X_train.T@error/len(X_train)
  db = np.sum(error)/len(X_train)

  weights -= learning_rate*dw
  bias -= learning_rate*db

In [13]:
z = X_test @ weights + bias

probability = sigmoid(z)

prediction = (probability >= 0.5).astype(int)

accuracy = np.mean(prediction == Y_test)
print("\n==============================")
print("Accuracy :", accuracy * 100, "%")
print("==============================")


Accuracy : 81.5 %


In [14]:
sample = X_test[0]

probability = sigmoid(sample@weights+bias)
predicted_class = int(probability>=0.5)

print("\nProbability :", probability)
print("Prediction  :", predicted_class)
print("Actual      :", int(Y_test[0]))



Probability : 0.818542641346157
Prediction  : 1
Actual      : 1
